<a href="https://colab.research.google.com/github/dkumar-tech/Modern-CPP-Programming/blob/master/RAGPDFAssistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


import os
os.kill(os.getpid(), 9)





In [1]:
import os
import gradio as gr

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


from langchain_huggingface import HuggingFaceEmbeddings

from langchain_google_genai import (
    ChatGoogleGenerativeAI
)

from langchain_community.vectorstores import Chroma

from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

from langchain_core.prompts import ChatPromptTemplate

from langchain.chains.combine_documents import (
    create_stuff_documents_chain
)

from langchain_core.prompts import ChatPromptTemplate

from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Gemini API Key: ")



vectorstore = None
retrieval_chain = None

def build_rag(pdf_files):

    global vectorstore
    global retrieval_chain

    try:
        # Load PDFs
        documents = []

        for pdf in pdf_files:
            loader = PyPDFLoader(pdf.name)
            docs = loader.load()
            documents.extend(docs)

        # Split documents
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=800,
            chunk_overlap=150
        )

        chunks = splitter.split_documents(documents)

        print(f"Loaded Pages: {len(documents)}")
        print(f"Created Chunks: {len(chunks)}")

        # Create embeddings
        embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
        )


        # Create vector store
        vectorstore = Chroma.from_documents(
          documents=chunks,
          embedding=embeddings,
          persist_directory="./chroma_db"
        )

        # Create retriever
        retriever = vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": 5}
        )

        # Create LLM
        llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0
        )




        # Prompt
        prompt = ChatPromptTemplate.from_template(
            """
            You are a PDF Assistant.

            Use only the information present in the context
            to answer the user's question.

            If the answer is not available in the context,
            say:
            "I could not find the answer in the uploaded PDFs."

            Context:
            {context}

            Question:
            {input}
            """
        )

        # Create chains
        document_chain = create_stuff_documents_chain(
            llm,
            prompt
        )

        retrieval_chain = create_retrieval_chain(
            retriever,
            document_chain
        )

        return f"✅ Successfully processed {len(pdf_files)} PDF(s)"

    except Exception as e:
        import traceback

        print("❌ Error in build_rag()")
        traceback.print_exc()

        return f"Error: {str(e)}"

    prompt = ChatPromptTemplate.from_template(
        """
        You are a PDF Assistant.

        Use only the information present in the context
        to answer the user's question.

        If the answer is not available in the context,
        say:
        "I could not find the answer in the uploaded PDFs."

        Context:
        {context}

        Question:
        {input}
        """
    )

    document_chain = create_stuff_documents_chain(
        llm,
        prompt
    )

    retrieval_chain = create_retrieval_chain(
        retriever,
        document_chain
    )

    return f" Successfully processed {len(pdf_files)} PDF(s)"

def ask_question(question):

    global retrieval_chain

    if retrieval_chain is None:
        return "Please upload and process PDFs first."

    try:

        response = retrieval_chain.invoke(
            {
                "input": question
            }
        )

        return response["answer"]

    except Exception as e:

        return f"Error: {str(e)}"

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """
        # 📚 LangChain RAG PDF Assistant

        Upload one or more PDFs and ask questions.

        Workflow:

        PDF Upload → Ingestion → Chunking →
        Embeddings → ChromaDB →
        Retrieval → Augmentation → LLM Response
        """
    )

    with gr.Row():

        pdf_upload = gr.File(
            file_count="multiple",
            file_types=[".pdf"],
            label="Upload PDFs"
        )

    process_btn = gr.Button("Process PDFs")

    status_box = gr.Textbox(
        label="Status"
    )

    process_btn.click(
        fn=build_rag,
        inputs=[pdf_upload],
        outputs=[status_box]
    )

    gr.Markdown("## Ask Questions")

    question = gr.Textbox(
        label="Question",
        placeholder="Ask something about the PDFs..."
    )

    answer = gr.Textbox(
        label="Answer",
        lines=10
    )

    ask_btn = gr.Button("Ask")

    ask_btn.click(
        fn=ask_question,
        inputs=[question],
        outputs=[answer]
    )


demo.launch(
    debug=True,
    share=True
)


Gemini API Key: ··········


/tmp/ipykernel_16545/3987835220.py:182: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a48d67e73d48f71370.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Loaded Pages: 30
Created Chunks: 79


...

/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Loaded Pages: 64
Created Chunks: 230


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a48d67e73d48f71370.gradio.live
